In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import sys
import json
import gc
import zipfile
from pathlib import Path
from typing import Dict, List
from collections import Counter

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from numpy.random import SeedSequence

from sklearn.metrics import (
    precision_recall_fscore_support,
    balanced_accuracy_score,
    matthews_corrcoef,
    f1_score,
    roc_auc_score,
)

from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
experiment_name = "standard_cnn"

drive_project_path = "/content/drive/MyDrive/colab/denseNet121"
drive_scripts_path = os.path.join(drive_project_path, "scripts")

drive_preprocessed = "/content/drive/MyDrive/colab/denseNet121/preprocessed"
drive_preprocessed_zip = os.path.join(drive_preprocessed, "preprocessed_384.zip")
drive_split_indices = os.path.join(drive_preprocessed, "split_indices.json")

exp_suffix = experiment_name
drive_output_path = os.path.join(drive_project_path, "output", exp_suffix)
drive_checkpoint_path = os.path.join(drive_project_path, "checkpoints", exp_suffix)

local_preprocessed_path = "/content/data"
local_scripts_path = "/content/scripts_cnn"
local_checkpoint_path = os.path.join("/content/checkpoints_cnn", exp_suffix)

for p in [
    local_preprocessed_path,
    local_scripts_path,
    local_checkpoint_path,
    drive_output_path,
    drive_checkpoint_path,
]:
    os.makedirs(p, exist_ok=True)

get_ipython().system(f"cp -r {drive_scripts_path}/* {local_scripts_path}/")
sys.path.insert(0, local_scripts_path)

In [3]:
from checkpoints import CheckpointManager, initialize_master_state
from plotting_utils import TrainingPlotter
from training_logic import ModelTrainer, determine_optimal_epochs
from training_utils import (
    NPYImageFolder,
    calculate_dataset_stats,
    create_transforms,
    calculate_class_weights,
    grouped_bootstrap_ci,
    aggregate_predictions_by_group,
    create_data_loaders,
    print_data_distribution,
    verify_data_splits,
    get_device_info,
    TransformSubset,
    create_slide_level_folds,
    validate_split_quality,
)
from config import get_config, save_config

In [4]:
config = get_config(experiment_name)
save_config(config, drive_output_path)

n_iterations = int(config["n_iterations"])
n_folds = int(config["n_folds"])
base_seed = int(config["random_seed"])

checkpoint_manager = CheckpointManager(
    local_checkpoint_dir=local_checkpoint_path,
    drive_checkpoint_dir=drive_checkpoint_path,
    n_folds=n_folds,
)

plotter = TrainingPlotter(drive_output_path, dpi=300)

checkpoint_manager.cleanup_temp_files(verbose=True)
checkpoint_manager.sync_from_drive(verbose=True)

device = get_device_info()

# ---------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------
preprocessed_dir = os.path.join(local_preprocessed_path, "preprocessed")

if not os.path.exists(preprocessed_dir):
    if not os.path.exists(drive_preprocessed_zip):
        raise FileNotFoundError(
            f"Preprocessed data not found at: {drive_preprocessed_zip}"
        )
    with zipfile.ZipFile(drive_preprocessed_zip, "r") as zf:
        zf.extractall(local_preprocessed_path)
else:
    print(f"Using existing preprocessed data at: {preprocessed_dir}")

metadata: Dict = {}
metadata_path = os.path.join(preprocessed_dir, "metadata.json")
if os.path.exists(metadata_path):
    with open(metadata_path, "r") as f:
        metadata = json.load(f)
    print(f"Loaded metadata from: {os.path.basename(metadata_path)}")
else:
    print("WARNING: metadata.json not found; using minimal metadata.")
    metadata = {
        "method": "unknown",
        "bit_depth": None,
        "image_size": config.get("image_size", None),
        "n_channels": config.get("channels", 1),
    }

# Consistency checks
cfg_bit = config.get("bit_depth", None)
meta_bit = metadata.get("bit_depth", None)
if cfg_bit is not None and meta_bit is not None and cfg_bit != meta_bit:
    print(
        f"WARNING: config bit_depth={cfg_bit} but preprocessing "
        f"bit_depth={meta_bit}."
    )

meta_size = metadata.get("image_size", None)
cfg_size = config.get("image_size", None)
if meta_size is not None and cfg_size is not None and meta_size != cfg_size:
    print(
        f"WARNING: config image_size={cfg_size} but preprocessing "
        f"image_size={meta_size}."
    )

  Config saved to: /content/drive/MyDrive/colab/denseNet121/output/standard_cnn/config_standard_cnn.json
  Synced: cv_iter_25_fold_2_checkpoint.pth
  Synced: master_training_state.json
Sync complete: 2 file(s) updated

Device Information:
  Device: cuda
  GPU: Tesla T4
  Memory: 15.64 GB
  CUDA Version: 12.8
Loaded metadata from: metadata.json


In [5]:
if not os.path.exists(drive_split_indices):
    raise FileNotFoundError(
        f"Split indices not found at: {drive_split_indices}\n"
        f"This file must be the IDENTICAL split used by all ViT experiments. "
        f"Do not regenerate it."
    )

with open(drive_split_indices, "r") as f:
    split_data = json.load(f)

train_base_paths = [Path(p) for p in split_data["train_base_paths"]]
test_base_paths = [Path(p) for p in split_data["test_base_paths"]]
class_names = split_data["classes"]

print("Classes:", class_names)
print("Train images:", len(train_base_paths))
print("Test images:", len(test_base_paths))

dataset = NPYImageFolder(preprocessed_dir)
num_classes = len(dataset.classes)

labels = np.array(dataset.targets)

print("Total images:", len(dataset))
print("Num classes:", num_classes)
print("Dataset classes:", dataset.classes)

npy_base_to_idx: Dict[str, int] = {}
idx_to_base: Dict[int, Path] = {}

for idx, (sample_path, target) in enumerate(dataset.samples):
    rel_npy = Path(sample_path).relative_to(preprocessed_dir)
    base_npy = rel_npy.with_suffix("")
    npy_base_to_idx[base_npy.as_posix()] = idx
    idx_to_base[idx] = Path(base_npy)

train_val_idx: List[int] = []
test_idx: List[int] = []

for base in train_base_paths:
    key = base.as_posix()
    if key not in npy_base_to_idx:
        raise RuntimeError(
            f"Train base path not found in dataset mapping: {key}"
        )
    train_val_idx.append(npy_base_to_idx[key])

for base in test_base_paths:
    key = base.as_posix()
    if key not in npy_base_to_idx:
        raise RuntimeError(
            f"Test base path not found in dataset mapping: {key}"
        )
    test_idx.append(npy_base_to_idx[key])

print("TrainVal indices:", len(train_val_idx))
print("Test indices:", len(test_idx))

verify_data_splits(train_val_idx, [], test_idx)
print_data_distribution(labels[train_val_idx].tolist(), dataset.classes, "TrainVal")
print_data_distribution(labels[test_idx].tolist(), dataset.classes, "Test")

Classes: ['Candida albicans', 'Candida glabrata']
Train images: 2910
Test images: 821
NPYImageFolder initialized: 3763 samples (grayscale)
Total images: 3763
Num classes: 2
Dataset classes: ['Candida albicans', 'Candida glabrata']
TrainVal indices: 2910
Test indices: 821
✓ Data splits verified: No overlap detected

TrainVal Distribution:
  Total samples: 2910
  Candida albicans: 1616 (55.5%)
  Candida glabrata: 1294 (44.5%)

Test Distribution:
  Total samples: 821
  Candida albicans: 440 (53.6%)
  Candida glabrata: 381 (46.4%)


In [6]:
def extract_slide_id_from_base(p: Path) -> str:
    """
    Extract slide identifier from patch base path.
    Verbatim from vit_fungi.py — must not be modified.
    """
    stem = p.name
    parts = stem.split("_")
    if len(parts) < 3:
        return stem
    date, species_code, letter = parts[0], parts[1], parts[2]
    return f"{date}_{species_code}_{letter}"


all_groups = np.empty(len(dataset), dtype=object)
for idx in range(len(dataset)):
    base_path = idx_to_base[idx]
    slide_id = extract_slide_id_from_base(base_path)
    all_groups[idx] = slide_id

test_groups = all_groups[test_idx]


# ---------------------------------------------------------------------------
# Phase 1: Cross-validation
# ---------------------------------------------------------------------------

def run_cross_validation(master_state: Dict) -> None:

    print("-" * 70)
    print("PHASE: CROSS-VALIDATION")
    print("-" * 70)
    print(f"Base seed: {base_seed}")

    start_iteration = int(master_state.get("current_iteration", 0))
    start_fold = int(master_state.get("current_fold", 0))
    cv_fold_details = list(master_state.get("cv_fold_details", []))

    trainer = ModelTrainer(config, device, checkpoint_manager, num_classes)

    ss = SeedSequence(base_seed)
    iteration_seed_gens = ss.spawn(n_iterations)
    iteration_seeds_base = [
        int(g.generate_state(1)[0]) for g in iteration_seed_gens
    ]

    trainval_idx_np = np.array(train_val_idx)

    for iter_idx in range(start_iteration, n_iterations):
        iteration_seed_base = iteration_seeds_base[iter_idx]
        print("\n" + "=" * 70)
        print(
            f"ITERATION {iter_idx + 1}/{n_iterations} | "
            f"base seed {iteration_seed_base}"
        )
        print("=" * 70)

        min_slides_per_class = config.get("min_slides_per_class_validation", 2)
        min_slide_balance_ratio = config.get("min_slide_balance_ratio", 0.4)
        max_image_imbalance_ratio = config.get("max_image_imbalance_ratio", 0.2)

        try:
            fold_splits, split_metadata = create_slide_level_folds(
                image_indices=trainval_idx_np,
                image_labels=labels,
                slide_groups=all_groups,
                n_splits=n_folds,
                random_state=iteration_seed_base,
                class_names=dataset.classes,
                min_slides_per_class=min_slides_per_class,
                min_slide_balance_ratio=min_slide_balance_ratio,
                max_image_imbalance_ratio=max_image_imbalance_ratio,
                max_retries=50,
                verbose=True,
            )

            iteration_seed_effective = split_metadata['random_state_effective']
            retry_count = split_metadata['retry_count']

            print(f"  Effective seed: {iteration_seed_effective}")
            if retry_count > 0:
                print(
                    f"  Required {retry_count} "
                    f"{'retry' if retry_count == 1 else 'retries'}"
                )

        except RuntimeError as e:
            print(f"\n FATAL ERROR: Could not create valid slide-stratified folds")
            print(f"\n{e}")
            print(
                f"\nAborting iteration {iter_idx + 1}. "
                f"Please review dataset and constraints."
            )

            master_state["current_iteration"] = iter_idx
            master_state["current_fold"] = 0
            master_state["split_error"] = str(e)
            checkpoint_manager.save_master_state(
                master_state, backup_to_drive=True
            )
            return

        if "split_metadata_per_iteration" not in master_state:
            master_state["split_metadata_per_iteration"] = {}

        master_state["split_metadata_per_iteration"][iter_idx] = {
            "iteration_seed_base": int(iteration_seed_base),
            "iteration_seed_effective": int(iteration_seed_effective),
            "retry_count": int(retry_count),
            "n_folds": n_folds,
            "min_slides_per_class": min_slides_per_class,
            "min_slide_balance_ratio": min_slide_balance_ratio,
            "fold_quality_stats": [
                {
                    'fold_idx': meta['fold_idx'],
                    'val_n_slides': meta['val_stats']['n_slides'],
                    'val_n_images': meta['val_stats']['n_images'],
                    'val_slide_balance': meta['val_stats']['slide_balance_ratio'],
                    'val_image_balance': meta['val_stats']['image_balance_ratio'],
                    'val_slides_per_class': meta['val_stats']['slides_per_class'],
                }
                for meta in split_metadata['fold_metadata']
            ],
        }

        for fold_idx, (cv_train_global, cv_val_global) in enumerate(fold_splits):
            if iter_idx == start_iteration and fold_idx < start_fold:
                continue

            print(f"Fold {fold_idx + 1}/{n_folds}")

            fold_meta = split_metadata['fold_metadata'][fold_idx]
            train_stats = fold_meta['train_stats']
            val_stats = fold_meta['val_stats']

            is_valid, error_msg = validate_split_quality(
                train_stats, val_stats, dataset.classes,
                min_slides_per_class, min_slide_balance_ratio,
                max_image_imbalance_ratio,
            )

            if not is_valid:
                raise RuntimeError(
                    f"Split validation failed for "
                    f"Iter {iter_idx+1} Fold {fold_idx+1}.\n"
                    f"This indicates a bug in create_slide_level_folds.\n"
                    f"Error: {error_msg}"
                )

            print(
                f"\nCalculating normalization statistics from training set..."
            )
            fold_mean, fold_std = calculate_dataset_stats(
                dataset=dataset,
                indices=cv_train_global.tolist(),
                batch_size=int(config["batch_size"]),
                num_workers=int(config["num_workers"]),
            )
            print(f"  Mean: {[f'{x:.4f}' for x in fold_mean]}")
            print(f"  Std:  {[f'{x:.4f}' for x in fold_std]}")

            train_transform = create_transforms(
                fold_mean, fold_std, config, train=True
            )
            val_transform = create_transforms(
                fold_mean, fold_std, config, train=False
            )

            train_loader, val_loader = create_data_loaders(
                dataset=dataset,
                train_indices=cv_train_global.tolist(),
                val_indices=cv_val_global.tolist(),
                train_transform=train_transform,
                val_transform=val_transform,
                batch_size=int(config["batch_size"]),
                num_workers=int(config["num_workers"]),
            )

            fold_class_weights = calculate_class_weights(
                targets=labels[cv_train_global].tolist(),
                num_classes=num_classes,
                device=device,
                gamma=float(config.get("class_weight_gamma", 1.0)),
            )

            val_groups_fold = all_groups[cv_val_global]

            if val_groups_fold is None or len(val_groups_fold) == 0:
                raise RuntimeError(
                    f"No slide groups found for validation fold {fold_idx+1}."
                )

            print(f"\nTraining fold {fold_idx + 1}...")
            fold_summary = trainer.train_cv_fold(
                iter_idx=iter_idx,
                fold_idx=fold_idx,
                iteration_seed=int(iteration_seed_effective),
                train_loader=train_loader,
                val_loader=val_loader,
                fold_mean=fold_mean,
                fold_std=fold_std,
                class_weights=fold_class_weights,
                val_groups_fold=val_groups_fold,
            )

            fold_summary["iteration_seed_base"] = int(iteration_seed_base)
            fold_summary["iteration_seed_effective"] = int(
                iteration_seed_effective
            )
            fold_summary["split_retry_count"] = int(retry_count)
            fold_summary["val_slides_per_class"] = val_stats['slides_per_class']
            fold_summary["val_slide_balance_ratio"] = val_stats[
                'slide_balance_ratio'
            ]
            fold_summary["val_image_balance_ratio"] = val_stats[
                'image_balance_ratio'
            ]

            cv_fold_details.append(fold_summary)

            master_state["cv_fold_details"] = cv_fold_details
            master_state["current_iteration"] = iter_idx
            master_state["current_fold"] = fold_idx + 1

            if fold_idx + 1 >= n_folds:
                master_state["current_iteration"] = iter_idx + 1
                master_state["current_fold"] = 0

            checkpoint_manager.save_master_state(
                master_state, backup_to_drive=True
            )
            checkpoint_manager.cleanup_previous_fold_checkpoint(
                iter_idx, fold_idx, verbose=True
            )

            del train_loader, val_loader, fold_class_weights
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        start_fold = 0

    master_state["all_iterations_completed"] = True
    master_state["phase"] = "analysis"
    checkpoint_manager.save_master_state(master_state, backup_to_drive=True)

    print("\n" + "=" * 70)
    print("CROSS-VALIDATION COMPLETE")
    print("=" * 70)
    print(f"\nCompleted {n_iterations} iterations × {n_folds} folds")
    print(f"Total fold summaries: {len(cv_fold_details)}")
    print(f"\nNext phase: Analysis")
    print("=" * 70 + "\n")

In [7]:
def run_analysis(master_state: Dict) -> None:

    print("-" * 70)
    print("PHASE: ANALYSIS")
    print("-" * 70)

    cv_fold_details = master_state.get("cv_fold_details", [])
    if not cv_fold_details:
        print("ERROR: No CV fold details found in master_state.")
        return

    cv_results_df = pd.DataFrame(cv_fold_details)
    n_folds_total = len(cv_results_df)
    n_iters_found = (
        cv_results_df["iteration"].nunique()
        if "iteration" in cv_results_df.columns else None
    )
    print(
        f"Loaded {n_folds_total} fold summaries | iterations={n_iters_found}"
    )

    results_csv_path = checkpoint_manager.get_results_csv_path(
        config["experiment_name"], use_local=True
    )
    cv_results_df.to_csv(results_csv_path, index=False)
    print(f"CV results saved to: {results_csv_path}")

    drive_csv_path = checkpoint_manager.get_results_csv_path(
        config["experiment_name"], use_local=False
    )
    try:
        import shutil
        shutil.copy2(results_csv_path, drive_csv_path)
        print(f"Backed up to Drive: {drive_csv_path}")
    except Exception as e:
        print(f"WARNING: Could not backup CSV to Drive: {e}")

    plotter.plot_iteration_results(
        cv_results_df,
        experiment_name=config["experiment_name"],
        save_name=f"cv_iteration_results_{config['experiment_name']}.png",
        show=False,
        prefer_slide_level=True,
    )

    print("Determining optimal number of epochs...")
    optimal_epochs = determine_optimal_epochs(
        cv_fold_details=cv_results_df.to_dict("records"),
        method="median",
        config=config,
    )
    print("Selected optimal_epochs:", optimal_epochs)

    master_state["optimal_epochs"] = int(optimal_epochs)

    master_state["phase"] = "final_training"
    checkpoint_manager.save_master_state(master_state, backup_to_drive=True)

    print("-" * 70)
    print("ANALYSIS COMPLETE")
    print("-" * 70)

In [8]:
def run_final_training(master_state: Dict) -> None:

    print("-" * 70)
    print("PHASE: FINAL TRAINING")
    print("-" * 70)

    optimal_epochs = int(
        master_state.get("optimal_epochs", config.get("num_epochs", 50))
    )
    print(f"Training final model for {optimal_epochs} epochs")

    print(
        "Calculating normalization statistics on full training set..."
    )
    final_mean, final_std = calculate_dataset_stats(
        dataset=dataset,
        indices=train_val_idx,
        batch_size=int(config["batch_size"]),
        num_workers=int(config["num_workers"]),
    )
    print("Final mean:", final_mean)
    print("Final std:", final_std)

    train_transform = create_transforms(final_mean, final_std, config, train=True)
    final_dataset = TransformSubset(dataset, list(train_val_idx), train_transform)

    final_loader = torch.utils.data.DataLoader(
        final_dataset,
        batch_size=int(config["batch_size"]),
        shuffle=True,
        num_workers=int(config["num_workers"]),
        pin_memory=True,
    )

    final_class_weights = calculate_class_weights(
        targets=labels[train_val_idx],
        num_classes=num_classes,
        device=device,
        gamma=float(config.get("class_weight_gamma", 1.0)),
    )

    trainer = ModelTrainer(config, device, checkpoint_manager, num_classes)
    final_model, final_training_history = trainer.train_final_model(
        train_loader=final_loader,
        optimal_epochs=optimal_epochs,
        final_mean=final_mean,
        final_std=final_std,
        class_weights=final_class_weights,
    )

    final_model_save_path = os.path.join(
        drive_output_path,
        f"final_model_{config['experiment_name']}.pth",
    )
    torch.save(
        {
            "model_state_dict": final_model.state_dict(),
            "config": config,
            "mean": final_mean,
            "std": final_std,
            "class_to_idx": dataset.class_to_idx,
            "optimal_epochs": int(optimal_epochs),
            "training_history": final_training_history,
            "metadata": metadata,
        },
        final_model_save_path,
    )
    print(f"Final model saved to: {final_model_save_path}")

    plotter.plot_training_curves(
        final_training_history,
        title=f"Final Model Training - {config['experiment_name']}",
        save_name=(
            f"final_model_training_curves_{config['experiment_name']}.png"
        ),
        show=False,
    )

    master_state["final_training_completed"] = True
    master_state["phase"] = "evaluation"
    checkpoint_manager.save_master_state(master_state, backup_to_drive=True)

    print("-" * 70)
    print("FINAL TRAINING COMPLETE")
    print("-" * 70)

In [9]:
def run_evaluation(master_state: Dict) -> None:

    print("\n" + "=" * 70)
    print("PHASE: TEST SET EVALUATION")
    print("=" * 70 + "\n")

    final_model_path = os.path.join(
        drive_output_path,
        f"final_model_{config['experiment_name']}.pth",
    )

    if not os.path.exists(final_model_path):
        print(f"ERROR: Final model not found at {final_model_path}")
        return

    print(
        f"  Loading final model from {os.path.basename(final_model_path)}"
    )
    checkpoint = torch.load(final_model_path, map_location=device)

    from densenet_model import DenseNet121Classifier

    model = DenseNet121Classifier(
        num_classes=num_classes,
        **{k: v for k, v in config["model_config"].items()
           if k != 'num_classes'},
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    print("  DenseNet121Classifier loaded successfully")

    counts = model.count_parameters()
    print(
        f"  Parameters: {counts['total']:,} total, "
        f"{counts['trainable']:,} trainable"
    )

    # Test data loader
    test_mean = checkpoint["mean"]
    test_std = checkpoint["std"]
    test_transform = create_transforms(test_mean, test_std, config, train=False)

    _, test_loader = create_data_loaders(
        dataset,
        train_indices=[],
        val_indices=test_idx,
        train_transform=None,
        val_transform=test_transform,
        batch_size=config["batch_size"],
        num_workers=config["num_workers"],
    )

    # Inference
    all_preds = []
    all_labels = []
    all_probs_list = []

    print(f"  Running inference on {len(test_idx)} test samples...")
    with torch.inference_mode():
        for X, y in tqdm(test_loader, desc="Testing", ncols=80, leave=False):
            X = X.to(device)
            logits = model(X)
            probs = logits.softmax(dim=1)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_probs_list.append(probs.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    print("  Inference complete")

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.vstack(all_probs_list)

    # ----------------------------------------------------------------------
    # 1) IMAGE-LEVEL METRICS
    # ----------------------------------------------------------------------
    print("\n  Computing image-level metrics...")
    test_bal_acc = balanced_accuracy_score(all_labels, all_preds)
    test_mcc = matthews_corrcoef(all_labels, all_preds)
    test_f1 = f1_score(
        all_labels, all_preds, average="weighted", zero_division=0
    )
    test_auc = roc_auc_score(all_labels, all_probs[:, 1])

    print(
        "  Calculating 95% confidence intervals (1000 bootstrap) "
        "— image level..."
    )
    bal_acc_ci = grouped_bootstrap_ci(
        balanced_accuracy_score,
        [all_labels, all_preds],
        groups=test_groups,
        n_bootstrap=1000,
        ci=0.95,
        seed=42,
        verbose=False,
    )
    mcc_ci = grouped_bootstrap_ci(
        matthews_corrcoef,
        [all_labels, all_preds],
        groups=test_groups,
        n_bootstrap=1000,
        ci=0.95,
        seed=43,
        verbose=False,
    )
    f1_ci = grouped_bootstrap_ci(
        lambda y_true, y_pred: f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        [all_labels, all_preds],
        groups=test_groups,
        n_bootstrap=1000,
        ci=0.95,
        seed=44,
        verbose=False,
    )
    auc_ci = grouped_bootstrap_ci(
        lambda y_true, p_hat: roc_auc_score(y_true, p_hat[:, 1]),
        [all_labels, all_probs],
        groups=test_groups,
        n_bootstrap=1000,
        ci=0.95,
        seed=45,
        verbose=False,
    )

    # ----------------------------------------------------------------------
    # 2) SLIDE-LEVEL AGGREGATION AND METRICS
    # ----------------------------------------------------------------------
    print("\n  Aggregating predictions to slide level...")
    (
        slide_true,
        slide_pred,
        slide_proba,
        slide_ids,
    ) = aggregate_predictions_by_group(
        y_true=all_labels,
        y_pred=all_preds,
        y_proba=all_probs,
        groups=test_groups,
        num_classes=num_classes,
    )

    print(
        f"  Aggregated {len(all_labels)} images into "
        f"{len(slide_true)} slides"
    )

    slide_bal_acc = balanced_accuracy_score(slide_true, slide_pred)
    slide_mcc = matthews_corrcoef(slide_true, slide_pred)
    slide_f1 = f1_score(
        slide_true, slide_pred, average="weighted", zero_division=0
    )
    slide_auc = roc_auc_score(slide_true, slide_proba[:, 1])

    print(
        "  Calculating 95% confidence intervals (1000 bootstrap) "
        "— slide level..."
    )
    slide_groups_dummy = np.arange(len(slide_true))

    slide_bal_acc_ci = grouped_bootstrap_ci(
        balanced_accuracy_score,
        [slide_true, slide_pred],
        groups=slide_groups_dummy,
        n_bootstrap=1000,
        ci=0.95,
        seed=52,
        verbose=False,
    )
    slide_mcc_ci = grouped_bootstrap_ci(
        matthews_corrcoef,
        [slide_true, slide_pred],
        groups=slide_groups_dummy,
        n_bootstrap=1000,
        ci=0.95,
        seed=53,
        verbose=False,
    )
    slide_f1_ci = grouped_bootstrap_ci(
        lambda y_true, y_pred: f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        [slide_true, slide_pred],
        groups=slide_groups_dummy,
        n_bootstrap=1000,
        ci=0.95,
        seed=54,
        verbose=False,
    )
    slide_auc_ci = grouped_bootstrap_ci(
        lambda y_true, p_hat: roc_auc_score(y_true, p_hat[:, 1]),
        [slide_true, slide_proba],
        groups=slide_groups_dummy,
        n_bootstrap=1000,
        ci=0.95,
        seed=55,
        verbose=False,
    )

    # ---------------------------------------------------------------------- #
    # 3) PER-CLASS METRICS
    # ---------------------------------------------------------------------- #
    print("\n  Computing per-class metrics...")

    p_img, r_img, f1_img, s_img = precision_recall_fscore_support(
        all_labels,
        all_preds,
        labels=list(range(num_classes)),
        average=None,
        zero_division=0,
    )

    per_class_metrics_image = {}
    for i, class_name in enumerate(dataset.classes):
        per_class_metrics_image[class_name] = {
            "precision": float(p_img[i]),
            "recall": float(r_img[i]),
            "f1_score": float(f1_img[i]),
            "support": int(s_img[i]),
        }

    p_slide, r_slide, f1_slide, s_slide = precision_recall_fscore_support(
        slide_true,
        slide_pred,
        labels=list(range(num_classes)),
        average=None,
        zero_division=0,
    )

    per_class_metrics_slide = {}
    for i, class_name in enumerate(dataset.classes):
        per_class_metrics_slide[class_name] = {
            "precision": float(p_slide[i]),
            "recall": float(r_slide[i]),
            "f1_score": float(f1_slide[i]),
            "support": int(s_slide[i]),
        }

    # ---------------------------------------------------------------------- #
    # 4) PRINT SUMMARY
    # ---------------------------------------------------------------------- #
    print("\n" + "=" * 70)
    print("TEST SET RESULTS SUMMARY")
    print("=" * 70)

    print("\nIMAGE-LEVEL METRICS (Patch-based):")
    print(
        f"  Balanced Accuracy: {test_bal_acc:.4f} "
        f"(95% CI: [{bal_acc_ci[0]:.4f}, {bal_acc_ci[1]:.4f}])"
    )
    print(
        f"  MCC:               {test_mcc:.4f} "
        f"(95% CI: [{mcc_ci[0]:.4f}, {mcc_ci[1]:.4f}])"
    )
    print(
        f"  F1-Score:          {test_f1:.4f} "
        f"(95% CI: [{f1_ci[0]:.4f}, {f1_ci[1]:.4f}])"
    )
    print(
        f"  AUC:               {test_auc:.4f} "
        f"(95% CI: [{auc_ci[0]:.4f}, {auc_ci[1]:.4f}])"
    )

    print("\nSLIDE-LEVEL METRICS (Aggregated):")
    print(
        f"  Balanced Accuracy: {slide_bal_acc:.4f} "
        f"(95% CI: [{slide_bal_acc_ci[0]:.4f}, {slide_bal_acc_ci[1]:.4f}])"
    )
    print(
        f"  MCC:               {slide_mcc:.4f} "
        f"(95% CI: [{slide_mcc_ci[0]:.4f}, {slide_mcc_ci[1]:.4f}])"
    )
    print(
        f"  F1-Score:          {slide_f1:.4f} "
        f"(95% CI: [{slide_f1_ci[0]:.4f}, {slide_f1_ci[1]:.4f}])"
    )
    print(
        f"  AUC:               {slide_auc:.4f} "
        f"(95% CI: [{slide_auc_ci[0]:.4f}, {slide_auc_ci[1]:.4f}])"
    )

    print("\nPER-CLASS METRICS:")
    for class_name in dataset.classes:
        img_metrics = per_class_metrics_image[class_name]
        slide_metrics_cls = per_class_metrics_slide[class_name]
        print(f"\n  {class_name}:")
        print(
            f"    Image-level: P={img_metrics['precision']:.3f}, "
            f"R={img_metrics['recall']:.3f}, "
            f"F1={img_metrics['f1_score']:.3f} "
            f"(n={img_metrics['support']} images)"
        )
        print(
            f"    Slide-level: P={slide_metrics_cls['precision']:.3f}, "
            f"R={slide_metrics_cls['recall']:.3f}, "
            f"F1={slide_metrics_cls['f1_score']:.3f} "
            f"(n={slide_metrics_cls['support']} slides)"
        )

    print("=" * 70 + "\n")

    # ---------------------------------------------------------------------- #
    # 5) VISUALIZATIONS
    # ---------------------------------------------------------------------- #
    print("  Generating visualizations...")

    plotter.plot_confusion_matrix(
        all_labels,
        all_preds,
        dataset.classes,
        title="Test Set Confusion Matrix (Image-Level)",
        save_name=(
            f"confusion_matrix_image_{config['experiment_name']}.png"
        ),
        normalize=True,
        show=False,
    )

    plotter.plot_confusion_matrix(
        slide_true,
        slide_pred,
        dataset.classes,
        title="Test Set Confusion Matrix (Slide-Level)",
        save_name=(
            f"confusion_matrix_slide_{config['experiment_name']}.png"
        ),
        normalize=True,
        show=False,
    )

    plotter.plot_roc_auc_curves(
        all_labels,
        all_probs,
        title=f"ROC Curve (Image-Level) - {config['experiment_name']}",
        save_name=(
            f"roc_auc_curve_image_{config['experiment_name']}.png"
        ),
        show=False,
    )

    plotter.plot_roc_auc_curves(
        slide_true,
        slide_proba,
        title=f"ROC Curve (Slide-Level) - {config['experiment_name']}",
        save_name=(
            f"roc_auc_curve_slide_{config['experiment_name']}.png"
        ),
        show=False,
    )

    # ---------------------------------------------------------------------- #
    # 6) SAVE RESULTS
    # ---------------------------------------------------------------------- #
    test_results_data = {
        "experiment_name": config["experiment_name"],
        "architecture": "DenseNet121",
        "parameter_count": counts,
        "metadata": metadata,

        "overall_metrics_image_level": {
            "balanced_accuracy": float(test_bal_acc),
            "mcc": float(test_mcc),
            "f1_score_weighted": float(test_f1),
            "auc_binary": float(test_auc),
        },
        "confidence_intervals_95_image_level": {
            "balanced_accuracy": (float(bal_acc_ci[0]), float(bal_acc_ci[1])),
            "mcc": (float(mcc_ci[0]), float(mcc_ci[1])),
            "f1_score_weighted": (float(f1_ci[0]), float(f1_ci[1])),
            "auc_binary": (float(auc_ci[0]), float(auc_ci[1])),
        },

        "overall_metrics_slide_level": {
            "balanced_accuracy": float(slide_bal_acc),
            "mcc": float(slide_mcc),
            "f1_score_weighted": float(slide_f1),
            "auc_binary": float(slide_auc),
        },
        "confidence_intervals_95_slide_level": {
            "balanced_accuracy": (
                float(slide_bal_acc_ci[0]), float(slide_bal_acc_ci[1])
            ),
            "mcc": (float(slide_mcc_ci[0]), float(slide_mcc_ci[1])),
            "f1_score_weighted": (
                float(slide_f1_ci[0]), float(slide_f1_ci[1])
            ),
            "auc_binary": (
                float(slide_auc_ci[0]), float(slide_auc_ci[1])
            ),
        },

        "per_class_metrics_image_level": per_class_metrics_image,
        "per_class_metrics_slide_level": per_class_metrics_slide,

        "test_set_size_images": len(test_idx),
        "test_set_size_slides": int(len(np.unique(test_groups))),

        "unique_test_slides": sorted(
            [str(sid) for sid in np.unique(test_groups)]
        ),
    }

    results_json_path = os.path.join(
        drive_output_path,
        f"test_results_{config['experiment_name']}.json",
    )

    with open(results_json_path, 'w') as f:
        json.dump(test_results_data, f, indent=4)

    print(f"\n  Results saved to: {os.path.basename(results_json_path)}")
    checkpoint_manager.cleanup_fold_histories(verbose=True)

    master_state["phase"] = "completed"
    checkpoint_manager.save_master_state(master_state, backup_to_drive=True)

    print("\n" + "=" * 70)
    print("EVALUATION COMPLETE")
    print("=" * 70 + "\n")

In [ ]:
def main() -> None:
    master_state = checkpoint_manager.load_master_state(prefer_drive=True)
    if master_state is None:
        print("No previous training state found. Initializing new state...")
        master_state = initialize_master_state()
        checkpoint_manager.save_master_state(
            master_state, backup_to_drive=True
        )
        print("Master state initialized")
    else:
        print(
            f"Resuming from saved state | "
            f"phase={master_state.get('phase', 'unknown')}"
        )

    try:
        phase = master_state.get("phase", "cross_validation")

        if phase == "cross_validation":
            run_cross_validation(master_state)
            phase = "analysis"

        if phase == "analysis":
            run_analysis(master_state)
            phase = "final_training"

        if phase == "final_training":
            run_final_training(master_state)
            phase = "evaluation"

        if phase == "evaluation":
            run_evaluation(master_state)
            phase = "completed"

        if phase == "completed":
            print("-" * 70)
            print("PIPELINE COMPLETED SUCCESSFULLY")
            print("-" * 70)
            print("Experiment:", config["experiment_name"])
            print("Results:", drive_output_path)

    except KeyboardInterrupt:
        print("Training interrupted by user.")
        print("Current phase:", master_state.get("phase", "unknown"))
        print("State saved - run again to resume.")
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
        print("Current phase:", master_state.get("phase", "unknown"))
        print("State saved - fix error and run again to resume.")


if __name__ == "__main__":
    main()

Loaded master state from: master_training_state.json
Resuming from saved state | phase=cross_validation
----------------------------------------------------------------------
PHASE: CROSS-VALIDATION
----------------------------------------------------------------------
Base seed: 42
MixUp enabled: alpha=0.2, prob=0.5

ITERATION 26/30 | base seed 2257503667

✓ Valid splits found on first attempt
  Effective seed: 2257503667
Fold 4/5

Calculating normalization statistics from training set...
  Mean: ['0.2372']
  Std:  ['0.0072']
Using uniform class weights (gamma=0, recommended with MixUp)

Training fold 4...

ITERATION 26/30 | FOLD 4/5
Iteration Seed: 2257503667
Validation: ~9 slides, 533 patches
Starting fold from scratch
Model initialization seed: 485949071
  DenseNet121Classifier created with seed 485949071: 6,949,634 params (6,949,634 trainable)

 BEST MODEL at epoch 1 (reason: first_epoch)
   slide_BA: 0.5000 | val_loss: 1.7713
Iter 26, Fold 4, Epoch 1/100 | train_loss: 0.5180 | tr